# Pamoka 11 - Agentas-į-agentą (A2A) protokolas


## Sąranka


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kas yra A2A protokolas?

**Agentų tarpusavio (A2A) protokolas** yra atviras standartas, leidžiantis dirbtinio intelekto agentams bendrauti,
atrasti vieni kitus ir bendradarbiauti — net kai jie sukuriami skirtinguose karkasuose arba talpinami
skirtingose paslaugose.

Pagrindinės sąvokos:

- **Atradimas** – Agentai paskelbia *Agento kortelę*, kuri aprašo jų galimybes, todėl kitiems agentams (arba orkestratoriams) lengva rasti tinkamą specialistą užduočiai.
- **Pranešimų perdavimas** – Agentai keičiasi struktūruotais pranešimais per bendrą protokolą, todėl vieno agento užklausą gali suprasti ir įvykdyti kitas agentas, neatsižvelgiant į jo vidinę implementaciją.
- **Užduoties gyvavimo ciklas** – A2A apibrėžia būsenas, tokias kaip *submitted*, *working*, *completed* ir *failed*, suteikdama orkestratoriui pilną matomumą, kaip progresuoja deleguota užduotis.

Šioje pamokoje mes simuliuojame A2A stiliaus bendradarbiavimą, sujungdami tris specializuotus kelionių agentus
į darbo eigą, kur kiekvienas agentas prisideda savo ekspertize ir perduoda rezultatus kitam.


## Specializuotų kelionių agentų kūrimas


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Daugiagentinis bendradarbiavimas per darbo eigą

Sujungiame tris agentus į nuoseklią darbo eigą, kuri atspindi A2A žinučių perdavimą:

1. **CurrencyExchangeAgent** gauna vartotojo užklausą ir pateikia valiutų gaires.
2. **ActivityPlannerAgent** gauna praturtintą kontekstą ir prideda veiklų rekomendacijas.
3. **TravelManagerAgent** sintezuoja abu įvestis į galutinį kelionės santrauką.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## A2A supratimas produkcinėje aplinkoje

Produkcinėje aplinkoje A2A protokolas atveria galingas tarp paslaugų scenarijus:

| Galimybė | Aprašymas |
|---|---|
| **Sąveika tarp karkasų** | Agentas, sukurtas naudojant vieną karkasą, gali deleguoti užduotis agentui, sukurtam naudojant bet kurį kitą A2A atitinkantį karkasą, leidžiantį tikrą tarp organizacijų sąveiką. |
| **Paslaugų ribos** | Agentai gali gyventi atskiruose mikroservisuose, debesų regionuose arba net skirtingose organizacijose, tuo pačiu sklandžiai bendradarbiaudami. |
| **Dinaminis aptikimas** | Orkestratorius gali vykdymo metu užklausti Agent Card registrą, kad surastų geriausiai tinkantį specialistą konkrečiai dalinei užduočiai. |
| **Srautinimas & push pranešimai** | A2A palaiko Server-Sent Events (SSE) realaus laiko pažangos atnaujinimams ir push pranešimus ilgai trunkančioms užduotims. |

Darbo eiga, kurią sukūrėme aukščiau, yra supaprastinta, procese vykdoma šio modelio versija. Tikroje
diegimo aplinkoje kiekvienas agentas atvertų HTTP galinį tašką, paskelbtų Agent Card ir komunikuotų
per A2A JSON-RPC protokolą.


## Santrauka

Šioje pamokoje sužinojote:

1. **Kas yra A2A protokolas** — atviras standartas agentų tarpusavio aptikimui, pranešimų siuntimui,
   ir užduočių valdymui.
2. **Kaip sukurti specializuotus agentus** — Valiutų keitimo agentą, Veiklos planuotojo agentą,
   ir Kelionių vadybininko orkestratorių.
3. **Kaip sujungti agentus į darbo eigą** — naudojant `WorkflowBuilder` modeliuoti nuoseklų
   pranešimų perdavimą tarp agentų.
4. **Kaip A2A veikia gamybinėje aplinkoje** — leidžiant bendradarbiavimą tarp skirtingų sistemų ir paslaugų su dinamišku aptikimu ir srautiniais atnaujinimais.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Atsakomybės apribojimas:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors stengiamės užtikrinti tikslumą, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba turėtų būti laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudotis profesionalia žmogaus atlikta vertimo paslauga. Mes neatsakome už jokius nesusipratimus ar neteisingas interpretacijas, kylančias dėl šio vertimo naudojimo.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
